### DPad Apprach (decoupled from MODHE code)

In [1]:
from sage.all import *

# --- timing for the thesis: wall clock per operation, seconds ---
from time import perf_counter

timings = []


def measure(label, fn, *args, **kwargs):
    t0 = perf_counter()
    out = fn(*args, **kwargs)
    seconds = perf_counter() - t0
    timings.append((label, seconds))
    print(f"[time] {label}: {seconds:.1f} s")
    return out


In [2]:
# ==========================================
# Extracted from ModHE to make this script standalone
# ==========================================

L = 7
MLWE = 2
POLDEG = 2^14
Berr = 10

# RNS Primes and Scaling Factors
q_list = vector((576460752340123649, 9007199282003969, 9007199284101121, 9007199318704129, 9007199338627073, 9007199343869953, 9007199350161409))
p = 9007199379521537
p_enc = 9007199254740992

moduli = [q_list[i] for i in range(L)] + [p]

Q = 1
for i in range(L):
    Q = Q * q_list[i]

p_inv_q = [p^(-1) % q_list[i] for i in range(L)]

# Define the polynomial ring R over rational numbers
R = QQ['x']
x = R.gen()
firreducible = x^POLDEG + 1

# Constants for ModUp and ModDown
q_hat = [0] * L
q_hat_inv = [0] * L
B = [0] * L

for j in range(L):
    q_hat[j] = Q / q_list[j]
    q_hat_inv[j] = q_hat[j]^(-1) % q_list[j]
    B[j] = q_hat_inv[j] * q_hat[j]

# Helper Functions
def random_between(j, k):
    """Generates a random integer between j and k."""
    return int(random()*(k-j+1)) + j

def poly_mod_prime(pol_a, mod):
    """Applies a modulo operation to every coefficient in a polynomial."""
    L_coeffs = list(pol_a)
    pol_b = 0
    for i in range(len(L_coeffs)):
        coeff = int(L_coeffs[i]) % mod
        pol_b = pol_b + coeff * x^i    
    return pol_b

def shares_in_mod(poly_mat, size, mod_list):
    """Splits a large polynomial into smaller RNS shares."""
    poly_shares = []
    for i in range(size):
        poly_shares.append(poly_mod_prime(poly_mat, mod_list[i]))
    return poly_shares

In [3]:
d = L      # DPad's full decomposition size (standard)
m = L      # DPad's smaller, sloppy decomposition size for Rank-Up
r = 2      # ModHE standard rank
r_temp = 3 # DPad temporary rank

In [4]:
# Modified from ModHE to support DPad-HE dynamic gadget sizes
def ModDown(b_share_p, b_shares_q):
    out_size = len(b_shares_q) # Dynamic sizing! No longer locked to L.
    a_shares_q = [0] * out_size
    result_q = [0] * out_size
    
    # Base conversion from B to C
    for i in range(out_size):
        a_shares_q[i] = b_share_p % q_list[i]
        result_q[i] = p_inv_q[i] * (b_shares_q[i] - a_shares_q[i]) % q_list[i]
        
    return result_q

# Extracted from auxiliary_functions.sage
def rescaling_shares(coeff_shares, level_in, level_out):
    coeff_shares_scaled = copy(coeff_shares)
    for l in range(level_in, level_out, -1):
        for j in range(l):
            temp = (q_list[l]^-1) % q_list[j]
            coeff_shares_scaled[j] = temp * (coeff_shares_scaled[j] - coeff_shares_scaled[l]) % q_list[j]
        coeff_shares_scaled[l] = 0
    return coeff_shares_scaled

def rescaling_ciphertext(ct_shares, level_in, level_out):
    ct_scaled_shares = [0] * (level_out + 1)
    
    for i in range(POLDEG):
        in_shares = []
        for j in range(level_in + 1):
            temp = int(ct_shares[j].coefficients(sparse=False)[i])
            in_shares.append(temp)
            
        out_shares = rescaling_shares(in_shares, level_in, level_out)
        for j in range(level_out + 1):
            ct_scaled_shares[j] = ct_scaled_shares[j] + int(out_shares[j]) * x^i
            
    return ct_scaled_shares

In [5]:
# Extracted from KeyGen.sage
def public_private_keygen():
    # Generation of secret polynomial vector s=[s0, s1, ...]
    s = matrix(R, 1, MLWE)
    for k in range(MLWE):
        s_k = 0
        for i in range(POLDEG-1):
            coeff = random_between(0,1)
            s_k = s_k + coeff * x^i
        s_k = s_k + x^(POLDEG-1)
        s[0,k] = s_k

    # Generation of error polynomial vector e=[e0, e1, ...] for encryption
    e = matrix(R, 1, MLWE)
    for k in range(MLWE):
        e_k = 0
        for i in range(POLDEG):
            coeff = random_between(0, Berr)
            e_k = e_k + coeff * x^i
        e[0,k] = e_k

    # Generate a random matrix [A]_mlwe*mlwe
    A = matrix(R, MLWE, MLWE)
    for l in range(MLWE):
        for k in range(MLWE):
            a_lk = 0
            for i in range(POLDEG):
                coeff = int(Q * random())
                a_lk = a_lk + coeff * x^i
            A[l, k] = a_lk

    b_pk = matrix(R, 1, MLWE)
    b_pk = -A * s.transpose() + e.transpose()

    key = [s, b_pk, A]
    print('private-public keypair generated')
    return key

# Extracted from Encrypt-Decrypt.sage
def encryption(m_plain, key):
    # Step1: generate error poly e0
    e0 = matrix(R, 1, MLWE)
    for k in range(MLWE):
        e_k = 0
        for i in range(POLDEG):
            coeff = random_between(0, Berr)
            e_k = e_k + coeff * x^i
        e0[0,k] = e_k

    # Step2: generate error poly e1
    e1 = matrix(R, 1, MLWE)
    for k in range(MLWE):
        e_k = 0
        for i in range(POLDEG):
            coeff = random_between(0, Berr)
            e_k = e_k + coeff * x^i
        e1[0,k] = e_k

    # Step3: generate sparse binary poly v
    v = matrix(R, 1, MLWE)
    for k in range(MLWE):
        v_k = 0
        for i in range(POLDEG):
            coeff = random_between(0,1) * random_between(0,1)
            v_k = v_k + coeff * x^i
        v[0,k] = v_k

    m_enc = round(m_plain * p_enc)
    print("m=", m_plain)

    c0_q = matrix(R, 1, 1)
    c1_q = matrix(R, 1, MLWE)

    b_pk = key[1]
    A = key[2]

    temp = (v * b_pk + m_enc + e0[0,0]) % firreducible
    c0_q[0,0] = poly_mod_prime(temp[0,0], Q)

    temp = (v * A + e1) % firreducible
    c1_q[0,0] = poly_mod_prime(temp[0,0], Q)
    c1_q[0,1] = poly_mod_prime(temp[0,1], Q)

    c0_qshares = shares_in_mod(c0_q[0,0], L, q_list)
    c10_qshares = shares_in_mod(c1_q[0,0], L, q_list)
    c11_qshares = shares_in_mod(c1_q[0,1], L, q_list)

    c1_qshares = [c10_qshares, c11_qshares]
    ciphertext = [c0_qshares, c1_qshares]

    return ciphertext

def decryption(ct, key, level1, level2):
    print("decryption:")
    c0_q = ct[0]
    c1_q = ct[1]

    s = key[0]
    s0 = s[0,0]
    s1 = s[0,1]

    if (level2 > level1):
        exit()

    if (level1 > level2):
        c0_dec_q_shares = rescaling_ciphertext(c0_q, level1, level2)
        c10_dec_q_shares = rescaling_ciphertext(c1_q[0], level1, level2)
        c11_dec_q_shares = rescaling_ciphertext(c1_q[1], level1, level2)

    if (level1 == level2):
        c0_dec_q_shares = c0_q
        c10_dec_q_shares = c1_q[0]
        c11_dec_q_shares = c1_q[1]

    temp1 = (c10_dec_q_shares[0] * s0) % firreducible 
    temp2 = (c11_dec_q_shares[0] * s1) % firreducible

    temp = (c0_dec_q_shares[0][0] + temp1 + temp2) % firreducible
    dec = poly_mod_prime(temp[0][0], q_list[0])
    
    dec_coeff = dec.coefficients()
    dec_m = float(dec_coeff[0] / p_enc)
    return dec_m

# Extracted from HMult.sage
def homomorphic_mult(ciphertext1, ciphertext2, level, key):
    c0_qshares1 = ciphertext1[0] 
    c1_qshares1 = ciphertext1[1]

    c0_qshares2 = ciphertext2[0] 
    c1_qshares2 = ciphertext2[1] 

    d0_qshares = []
    d10_qshares = []
    d11_qshares = []
    d20_qshares = []
    d21_qshares = []
    d3_qshares = []

    for j in range(level + 1):
        temp = (c0_qshares1[j] * c0_qshares2[j]) % firreducible
        d0_qshares.append(poly_mod_prime(temp, q_list[j]))

    for j in range(level + 1):
        temp = (c0_qshares1[j] * c1_qshares2[0][j] + c1_qshares1[0][j] * c0_qshares2[j]) % firreducible
        d10_qshares.append(poly_mod_prime(temp, q_list[j]))

    for j in range(level + 1):
        temp = (c0_qshares1[j] * c1_qshares2[1][j] + c1_qshares1[1][j] * c0_qshares2[j]) % firreducible
        d11_qshares.append(poly_mod_prime(temp, q_list[j]))

    for j in range(level + 1):
        temp = (c1_qshares1[0][j] * c1_qshares2[0][j]) % firreducible
        d20_qshares.append(poly_mod_prime(temp, q_list[j]))

    for j in range(level + 1):
        temp = (c1_qshares1[1][j] * c1_qshares2[1][j]) % firreducible
        d21_qshares.append(poly_mod_prime(temp, q_list[j]))

    for j in range(level + 1):
        temp = (c1_qshares1[0][j] * c1_qshares2[1][j] + c1_qshares1[1][j] * c1_qshares2[0][j]) % firreducible
        d3_qshares.append(poly_mod_prime(temp, q_list[j]))

    d1_qshares = [d10_qshares, d11_qshares]
    d2_qshares = [d20_qshares, d21_qshares]
    
    return [d0_qshares, d1_qshares, d2_qshares, d3_qshares]

In [6]:
def generate_dpad_keys(key):
    print("[DEBUG] Generating True Cryptographic DPad-HE Keys from scratch...")
    
    # Extract the original Rank 2 secret key [s0, s1]
    s = key[0] 
    s0 = s[0,0]
    s1 = s[0,1]
    
    # --- 1. Generate s' (The temporary Rank-Up secret) ---
    # We sample a random binary polynomial exactly like the original secret
    s_prime_coeffs = [random_between(0,1) for _ in range(POLDEG-1)] + [1]
    s_prime = R(s_prime_coeffs)
    
    # Create the temporarily expanded secret basis [s0, s1, s']
    s_expand = matrix(R, 1, 3, [s0, s1, s_prime])
    
    crlk_b_array = []
    crlk_A_array = []
    
    # The three quadratic payloads we need to relinearize
    payloads = [s0*s0, s1*s1, s0*s1]
    
    print("  -> Building Cross-Relinearization Key (crlk)...")
    # --- 2. Generate crlk (Encrypts payloads under expanded basis) ---
    for payload in payloads:
        b_shares = []
        A_shares = []
        for j in range(L): # Loop over RNS modules
            # Sample random public matrix A (1x3 vector)
            A_vec = matrix(R, 1, 3)
            for v in range(3):
                A_vec[0, v] = R([int(p*Q*random()) for _ in range(POLDEG)])
            
            # Sample Gaussian error e (scalar)
            e_poly = R([random_between(0, Berr) for _ in range(POLDEG)])
            
            # Compute MLWE public key: b = -A * s_expand^T + e + p*B[j]*payload
            b_poly = (-A_vec * s_expand.transpose())[0,0] + e_poly + p*B[j]*payload
            b_poly = b_poly % firreducible
            
            # Convert to RNS shares for the Key Switch pipeline
            b_share_list = shares_in_mod(b_poly, L+1, moduli)
            A_share_list = []
            for i in range(L+1):
                A_share_list.append([
                    poly_mod_prime(A_vec[0,0], moduli[i]),
                    poly_mod_prime(A_vec[0,1], moduli[i]),
                    poly_mod_prime(A_vec[0,2], moduli[i])
                ])
            b_shares.append(b_share_list)
            A_shares.append(A_share_list)
            
        crlk_b_array.append(b_shares)
        crlk_A_array.append(A_shares)
        
    print("  -> Building Rank-Down Key (rdk)...")
    # --- 3. Generate rdk (Encrypts s' under original basis) ---
    rdk_b = []
    rdk_A = []
    for j in range(L):
        # Sample random public matrix A (1x2 vector)
        A_vec = matrix(R, 1, 2)
        for v in range(2):
            A_vec[0, v] = R([int(p*Q*random()) for _ in range(POLDEG)])
            
        # Sample Gaussian error e
        e_poly = R([random_between(0, Berr) for _ in range(POLDEG)])
        
        # Compute MLWE public key: b = -A * s^T + e + p*B[j]*s'
        b_poly = (-A_vec * s.transpose())[0,0] + e_poly + p*B[j]*s_prime
        b_poly = b_poly % firreducible
        
        b_share_list = shares_in_mod(b_poly, L+1, moduli)
        A_share_list = []
        for i in range(L+1):
            A_share_list.append([
                poly_mod_prime(A_vec[0,0], moduli[i]),
                poly_mod_prime(A_vec[0,1], moduli[i])
            ])
            
        rdk_b.append(b_share_list)
        rdk_A.append(A_share_list)
        
    print("[DEBUG] Keys successfully generated!")
    return crlk_b_array, crlk_A_array, rdk_b, rdk_A

In [7]:
def ModularKeySwitch(c_target, swk_b, swk_A, decomp_size):
    out_rank = len(swk_A[0][0]) 
    in_shares = len(c_target) 
    
    print(f"\n  [DEBUG - ModularKeySwitch] Starting...")
    print(f"  [DEBUG - ModularKeySwitch] decomp_size param: {decomp_size}")
    print(f"  [DEBUG - ModularKeySwitch] in_shares (len of c_target): {in_shares}")
    print(f"  [DEBUG - ModularKeySwitch] out_rank: {out_rank}")
    print(f"  [DEBUG - ModularKeySwitch] swk_b shape: {len(swk_b)} rows, {len(swk_b[0])} cols")
    
    # Debug check:
    if in_shares > len(swk_b):
        print(f"  [FATAL DEBUG WARN] in_shares ({in_shares}) is greater than swk_b rows ({len(swk_b)})! Loop 'j' will crash!")
        print(f"  [DEBUG] Type of c_target[0]: {type(c_target[0])}")

    c_new_0_primeshares = [0] * (in_shares + 1)
    c_new_1_primeshares = [[0] * out_rank for _ in range(in_shares + 1)]

    # Phase 1: Mod Up 
    for i in range(in_shares + 1):
        for j in range(in_shares):
            temp_0 = (c_target[j] * swk_b[j][i]) % firreducible
            c_new_0_primeshares[i] = poly_mod_prime(c_new_0_primeshares[i] + temp_0, moduli[i])
            
            for v in range(out_rank):
                temp_1 = (c_target[j] * swk_A[j][i][v]) % firreducible
                c_new_1_primeshares[i][v] = poly_mod_prime(c_new_1_primeshares[i][v] + temp_1, moduli[i])

    print("  [DEBUG - ModularKeySwitch] Mod Up completed successfully. Starting Mod Down...")
    
    def get_padded_coeffs(poly):
        if isinstance(poly, (int, Integer)):
            c = [int(poly)] if poly != 0 else []
        else:
            c = poly.coefficients(sparse=False)
        return c + [0] * (POLDEG - len(c))

    coeffs_0 = [get_padded_coeffs(p) for p in c_new_0_primeshares]
    coeffs_1 = [[get_padded_coeffs(p) for p in row] for row in c_new_1_primeshares]
    
    # Phase 2: Mod Down
    scaled_c0 = [0] * in_shares
    for k in range(POLDEG):
        shares_q = [coeffs_0[j][k] for j in range(in_shares)]
        share_p = coeffs_0[in_shares][k]
        scaled = ModDown(share_p, shares_q)
        for j in range(in_shares):
            scaled_c0[j] += int(scaled[j]) * x^k

    scaled_c1 = []
    for v in range(out_rank):
        scaled_v = [0] * in_shares
        for k in range(POLDEG):
            shares_q = [coeffs_1[j][v][k] for j in range(in_shares)]
            share_p = coeffs_1[in_shares][v][k]
            scaled = ModDown(share_p, shares_q)
            for j in range(in_shares):
                scaled_v[j] += int(scaled[j]) * x^k
        scaled_c1.append(scaled_v)

    return scaled_c0, scaled_c1

In [8]:
def DPad_Relin(c0, c1, c2_array, crlk_b_array, crlk_A_array, rdk_b, rdk_A, m, d):
    in_shares = len(c0)
    print(f"\n[DEBUG - DPad_Relin] in_shares (len of c0): {in_shares}")
    print(f"[DEBUG - DPad_Relin] c2_array length: {len(c2_array)}")
    
    total_c_hat_0 = [0] * in_shares
    total_c_hat_1 = [[0] * in_shares for _ in range(r_temp)] 
    
    for idx, c2_item in enumerate(c2_array):
        print(f"\n[DEBUG - DPad_Relin] Processing c2_item index {idx}. Length: {len(c2_item)}")
        
        c_hat_0, c_hat_1 = ModularKeySwitch(c2_item, crlk_b_array[idx], crlk_A_array[idx], m)
        
        for i in range(in_shares):
            total_c_hat_0[i] = poly_mod_prime(total_c_hat_0[i] + c_hat_0[i], moduli[i])
            for v in range(r_temp):
                total_c_hat_1[v][i] = poly_mod_prime(total_c_hat_1[v][i] + c_hat_1[v][i], moduli[i])
    
    overflow_part = total_c_hat_1[2]
    c_tilde_0, c_tilde_1 = ModularKeySwitch(overflow_part, rdk_b, rdk_A, d)
    
    final_c0 = [0] * in_shares
    final_c1 = [[0] * in_shares for _ in range(r)]
    
    for i in range(in_shares):
        final_c0[i] = poly_mod_prime(c0[i] + total_c_hat_0[i] + c_tilde_0[i], moduli[i])
        final_c1[0][i] = poly_mod_prime(c1[0][i] + total_c_hat_1[0][i] + c_tilde_1[0][i], moduli[i])
        final_c1[1][i] = poly_mod_prime(c1[1][i] + total_c_hat_1[1][i] + c_tilde_1[1][i], moduli[i])
        
    return final_c0, final_c1

In [9]:
print("\nGenerating Real Keys (this will take a moment)...")
# We only use ModHE to generate the basic secret/public keys for standard encryption
key = measure("keygen", public_private_keygen)

# We natively generate the DPad-HE architecture keys from scratch!
crlk_b, crlk_A, rdk_b, rdk_A = measure("dpad_keygen", generate_dpad_keys, key)


Generating Real Keys (this will take a moment)...
private-public keypair generated
[time] keygen: 36.1 s
[DEBUG] Generating True Cryptographic DPad-HE Keys from scratch...
  -> Building Cross-Relinearization Key (crlk)...
  -> Building Rank-Down Key (rdk)...
[DEBUG] Keys successfully generated!
[time] dpad_keygen: 752.8 s


In [10]:
m_plain1 = 0.4
m_plain2 = 0.6

print("Encrypting data...")
ct1 = measure("encrypt_1", encryption, m_plain1, key)
ct2 = measure("encrypt_2", encryption, m_plain2, key)

Encrypting data...
m= 0.400000000000000
[time] encrypt_1: 56.4 s
m= 0.600000000000000
[time] encrypt_2: 56.6 s


In [11]:
print("Executing Homomorphic Multiplication...")
mult_result = measure("hmult", homomorphic_mult, ct1, ct2, L-1, key)

Executing Homomorphic Multiplication...
[time] hmult: 39.7 s


In [12]:
print(f"\n[DEBUG - Execution] mult_result length: {len(mult_result)}")
d0 = mult_result[0]
d1_vector = mult_result[1] 
d2_vector_part = mult_result[2]
d3_scalar = mult_result[3]

print(f"[DEBUG - Execution] d0 length: {len(d0)}, type of d0[0]: {type(d0[0])}")
print(f"[DEBUG - Execution] d1_vector length: {len(d1_vector)}, length of d1_vector[0]: {len(d1_vector[0])}")
print(f"[DEBUG - Execution] d2_vector_part length: {len(d2_vector_part)}, length of d2_vector_part[0]: {len(d2_vector_part[0])}")
print(f"[DEBUG - Execution] d3_scalar length: {len(d3_scalar)}, type of d3_scalar[0]: {type(d3_scalar[0])}")

c0_input = d0
c1_input = d1_vector 
c2_input = [d2_vector_part[0], d2_vector_part[1], d3_scalar] 

print(f"[DEBUG - Execution] c2_input bundled. Length: {len(c2_input)}. Length of first item inside it: {len(c2_input[0])}")

print("\nExecuting DPad Relinearization...")
final_c0, final_c1 = measure("dpad_relin", DPad_Relin, c0_input, c1_input, c2_input, crlk_b, crlk_A, rdk_b, rdk_A, m, d)

print("\n--- DPAD PIPELINE COMPLETE ---")


[DEBUG - Execution] mult_result length: 4
[DEBUG - Execution] d0 length: 7, type of d0[0]: <class 'sage.rings.polynomial.polynomial_rational_flint.Polynomial_rational_flint'>
[DEBUG - Execution] d1_vector length: 2, length of d1_vector[0]: 7
[DEBUG - Execution] d2_vector_part length: 2, length of d2_vector_part[0]: 7
[DEBUG - Execution] d3_scalar length: 7, type of d3_scalar[0]: <class 'sage.rings.polynomial.polynomial_rational_flint.Polynomial_rational_flint'>
[DEBUG - Execution] c2_input bundled. Length: 3. Length of first item inside it: 7

Executing DPad Relinearization...

[DEBUG - DPad_Relin] in_shares (len of c0): 7
[DEBUG - DPad_Relin] c2_array length: 3

[DEBUG - DPad_Relin] Processing c2_item index 0. Length: 7

  [DEBUG - ModularKeySwitch] Starting...
  [DEBUG - ModularKeySwitch] decomp_size param: 7
  [DEBUG - ModularKeySwitch] in_shares (len of c_target): 7
  [DEBUG - ModularKeySwitch] out_rank: 3
  [DEBUG - ModularKeySwitch] swk_b shape: 7 rows, 8 cols
  [DEBUG - Modular

In [13]:
dpad_relin_result = [final_c0, final_c1]

print("Decrypting DPad result...")
final_HE_dec = measure("decrypt_after_rescale", decryption, dpad_relin_result, key, L-1, L-2)

print("\n--- RESULTS ---")
print(f"Expected Result (0.4 * 0.6): {m_plain1 * m_plain2}")
print(f"DPad Computed Result:        {final_HE_dec}")

Decrypting DPad result...
decryption:
[time] decrypt_after_rescale: 3846.5 s

--- RESULTS ---
Expected Result (0.4 * 0.6): 0.240000000000000
DPad Computed Result:        0.23999999744473577


In [14]:
# --- write the timing table: one row per operation, parameters repeated ---
import csv
import os
from datetime import datetime

path = "sagemath_timing.csv"
params = [("prototype", "dpad_prototype"), ("L", L), ("r", r), ("r_temp", r_temp), ("d", d), ("m", m), ("POLDEG", POLDEG)]
new_file = not os.path.exists(path)
stamp = datetime.now().strftime("%Y-%m-%d %H:%M")

with open(path, "a", newline="") as fh:
    writer = csv.writer(fh)
    if new_file:
        writer.writerow([name for name, _ in params] + ["operation", "seconds", "run"])
    for label, seconds in timings:
        writer.writerow([value for _, value in params] + [label, f"{seconds:.2f}", stamp])

print("\n--- TIMING (one run, this machine) ---")
for label, seconds in timings:
    print(f"{label:26s} {seconds:8.1f} s")
print(f"written to {path}")



--- TIMING (one run, this machine) ---
keygen                         36.1 s
dpad_keygen                   752.8 s
encrypt_1                      56.4 s
encrypt_2                      56.6 s
hmult                          39.7 s
dpad_relin                    978.0 s
decrypt_after_rescale        3846.5 s
written to sagemath_timing.csv
